# 中音谱号 Logo

A 精致传统（上下对称、粗腹细回转、圆团回钩）与 B 自由水墨（非对称、笔锋起收、按 seed 轻微手抖）都是手绘的闭合三次 Bézier 轮廓，C 的尖凹落在第三线 y = 0。

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.patches import PathPatch
from matplotlib.textpath import TextPath
from matplotlib.transforms import Affine2D

import clef

## 参数

越界会抛 `ValueError`，范围见 README。

In [ ]:
p = clef.Params(
    height=4.0, width=2.76, weight=0.25, opening=0.25, gap=0.25,
    contrast=0.6, seed=7, dryness=0.35, bleed=0.3,
)
PALETTE = "sumi"  # 展示用：ink / teal / cobalt / sumi
print(p)

## 官方参考对照

参考 [SMuFL cClef U+E05C](https://smufl.formats.music/latest/tables/clefs.html)。本机装有 Bravura 时并排显示；项目不打包、不配置字体。

In [ ]:
def staff(ax, x0, x1):
    for k in range(-2, 3):
        ax.plot([x0, x1], [k * p.height / 4] * 2, color="#C8402A" if k == 0 else "#9AA0A6", lw=1.2 if k == 0 else 0.6, zorder=3)

try:
    bravura = findfont(FontProperties(family="Bravura"), fallback_to_default=False)
except ValueError:
    bravura = None
panels = [("Bravura U+E05C", None)] if bravura else []
panels += [(label, clef.silhouette(fn(p))) for label, fn in clef.VARIANTS.values()]
fig, axes = plt.subplots(1, len(panels), figsize=(3.2 * len(panels), 3.6), squeeze=False)
bbox = clef.common_bbox(p)
for ax, (label, g) in zip(axes[0], panels):
    if g is None:
        tp = TextPath((0, 0), "\ue05c", size=1, prop=FontProperties(fname=bravura))
        v = tp.vertices
        to_staff = Affine2D().translate(-v[:, 0].min(), -(v[:, 1].max() + v[:, 1].min()) / 2).scale(p.height / np.ptp(v[:, 1]))
        ax.add_patch(PathPatch(to_staff.transform_path(tp), fc="#111111", lw=0))
        ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3]); ax.set_aspect("equal"); ax.axis("off")
    else:
        clef.draw_glyph(ax, g, clef.MONO, bbox)
    staff(ax, bbox[0], bbox[2])
    ax.set_title(label)
plt.show()
if not bravura:
    print("未找到 Bravura 字体：请打开上面的官方链接对照。")

## 黑剪影

去掉飞白与墨晕，只看轮廓。

In [ ]:
display(clef.silhouette_figure(p))

## 拆解

圆点 = 尖凹顶点（第三线），叉 = 卷叶留白探测点；第四列红色为飞白子路径。

In [ ]:
display(clef.breakdown_figure(p, PALETTE))

## 两方案同尺度

In [ ]:
fig, axes = plt.subplots(1, len(clef.VARIANTS), figsize=(9, 4.6))
for ax, (label, fn) in zip(axes, clef.VARIANTS.values()):
    clef.draw_glyph(ax, fn(p), PALETTE, bbox)
    ax.set_title(label)
plt.show()

## 四配色 + 单色

In [ ]:
display(clef.compare_figure(p))

## 参数小扫描

每行只改一个参数；同一方案的所有格子共用一个取景范围（全部结果的并集），大小可直接比较。

In [ ]:
sweeps = {
    "opening": [0.0, 0.25, 0.5, clef.OPENING_MAX],
    "weight": [0.15, 0.2, 0.25, 0.32],
    "contrast": [0.0, 0.4, 0.7, 1.0],
    "dryness": [0.0, 0.35, 0.7, 1.0],
    "bleed": [0.0, 0.3, 0.6, 1.0],
    "seed": [7, 8, 9, 10],
}
base = {f: getattr(p, f) for f in clef.Params.__dataclass_fields__}
for label, fn in clef.VARIANTS.values():
    cells = {}
    for name, values in sweeps.items():
        for v in values:
            try:
                cells[name, v] = fn(clef.Params(**{**base, name: v}))
            except ValueError as err:
                cells[name, v] = err
    shared = clef.square_bbox(clef.union_bbox(clef.glyph_bbox(g) for g in cells.values() if not isinstance(g, ValueError)))
    fig, axes = plt.subplots(len(sweeps), 4, figsize=(8, 2.1 * len(sweeps)))
    fig.suptitle(f"{label} · shared frame")
    for row, (name, values) in zip(axes, sweeps.items()):
        for ax, v in zip(row, values):
            g = cells[name, v]
            if isinstance(g, ValueError):
                ax.text(0.5, 0.5, f"{name}={v:g}\nout of range", ha="center", va="center", fontsize=8)
                ax.axis("off")
                continue
            clef.draw_glyph(ax, g, PALETTE, shared)
            ax.set_title(f"{name}={v:g}", fontsize=9)
    fig.tight_layout()
    plt.show()

## 32 / 64 px 与大图

小图是按真实像素栅格化后放大显示。

In [ ]:
fig, axes = plt.subplots(2, 2 * len(clef.VARIANTS), figsize=(9, 4.8))
for col, (label, fn) in enumerate(clef.VARIANTS.values()):
    g = fn(p)
    for row, px in enumerate((32, 64)):
        for j, pal in enumerate((PALETTE, clef.MONO)):
            ax = axes[row, 2 * col + j]
            ax.imshow(clef.rasterize(g, pal, px), interpolation="nearest")
            ax.set_title(f"{label[0]} @ {px}px", fontsize=9)
            ax.axis("off")
plt.show()

for label, fn in clef.VARIANTS.values():
    display(clef.render(fn(p), PALETTE, size=6))

## 导出

用上面的参数、ink 配色写出 `out/variant_{a,b}.{png,svg}`、`out/compare.png`、`out/strokes_breakdown.png`；默认参数时与 `uv run python clef.py` 相同。

In [ ]:
written = clef.export_all("out", p)
for name, path in written.items():
    print(f"{name:24s} {path.stat().st_size:>8d} bytes")